In [94]:
import visualkeras
from PIL import ImageFont
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization, MaxPooling2D, Flatten, Dense, InputLayer

if __name__ == "__main__":
    model = Sequential([
        InputLayer(input_shape=(18, 29, 29)),
        Conv2D(64, kernel_size=3, padding='same', data_format='channels_last'),
        BatchNormalization(axis=1),
        MaxPooling2D(pool_size=(2, 2), data_format='channels_last'),
        Conv2D(128, kernel_size=3, padding='same', data_format='channels_last'),
        BatchNormalization(axis=1),
        MaxPooling2D(pool_size=(2, 2), data_format='channels_last'),
        Conv2D(256, kernel_size=3, padding='same', data_format='channels_last'),
        BatchNormalization(axis=1),
        MaxPooling2D(pool_size=(2, 2), data_format='channels_last'),
        Conv2D(512, kernel_size=3, padding='same', data_format='channels_last'),
        BatchNormalization(axis=1),
        MaxPooling2D(pool_size=(2, 2), data_format='channels_last'),
        Flatten(),
        Dense(256, activation='relu'),
        Dense(128, activation='relu'),
        Dense(64, activation='relu'),
        Dense(2, activation=None),
    ])

    # model.build(input_shape=(None, 18, 29, 29))

    # dummy = tensorflow.zeros((1, 18, 29, 29))  
    # _ = model(dummy) 
    
    font = ImageFont.truetype("../times-new-roman/times.ttf", size=60)
    #model.add(visualkeras.SpacingDummyLayer(spacing=100))

    img = visualkeras.layered_view(
        model,
        legend=True,                             # 顯示圖例
        draw_volume=True,                        # 在每個方塊上標註輸出維度
        # show_dimension=True,                   # 顯示每個層的維度
        font=font,
        spacing=35,                              # 層與層之間的間距
        scale_xy=30,                              # 縮放比例
        scale_z=0.5,                               # Z軸縮放比例
    ).show()




In [103]:
import math
import visualkeras
from PIL import ImageFont
import tensorflow as tf
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    Conv2DTranspose,
    LeakyReLU,
    Flatten,
    Dense,
    Concatenate,
    Reshape,
    Cropping2D
)
from tensorflow.keras.models import Model

# -------------------------------
# 參數設定（請依實際需求調整）
# -------------------------------
in_ch       = 18    # 輸入通道數 (原 PyTorch 裡的 in_ch)
latent_dim  = 50    # 潛在空間維度
num_classes = 2     # 條件標籤數量
H, W        = 29, 29

# 計算經過三次 stride=2 後的 Feature map 大小 (ceil 版)
ds_h = math.ceil(H / 8)  # ceil(29/8) = 4
ds_w = math.ceil(W / 8)  # = 4
flattened_size = 256 * ds_h * ds_w  # 256 * 4 * 4 = 4096

# -------------------------------
# 1. 定義 Encoder
# -------------------------------
# x 輸入：形狀 (H, W, in_ch)；c 輸入：形狀 (num_classes,)
x_input = Input(shape=(H, W, in_ch), name="x_input")
c_input = Input(shape=(num_classes,), name="c_input")

# (A) 三層 Conv2D + LeakyReLU (與 PyTorch seq 一致)
h = Conv2D(
    filters=64,
    kernel_size=3,
    strides=2,
    padding="same",
    name="enc_conv1"
)(x_input)
h = LeakyReLU(alpha=0.2, name="enc_lrelu1")(h)

h = Conv2D(
    filters=128,
    kernel_size=3,
    strides=2,
    padding="same",
    name="enc_conv2"
)(h)
h = LeakyReLU(alpha=0.2, name="enc_lrelu2")(h)

h = Conv2D(
    filters=256,
    kernel_size=3,
    strides=2,
    padding="same",
    name="enc_conv3"
)(h)
h = LeakyReLU(alpha=0.2, name="enc_lrelu3")(h)
# 現在 h.shape = (None, ds_h=4, ds_w=4, 256)

# (B) Flatten 並和條件向量串接
h = Flatten(name="enc_flatten")(h)  # → (None, 256*4*4 = 4096)
h = Concatenate(name="enc_concat")([h, c_input])  # → (None, 4096 + num_classes)

# (C) 全連接層 fc_enc 與 LeakyReLU
h = Dense(512, name="enc_fc1")(h)
h = LeakyReLU(alpha=0.2, name="enc_lrelu4")(h)
h = Dense(256, name="enc_fc2")(h)
h = LeakyReLU(alpha=0.2, name="enc_lrelu5")(h)

# (D) 分別輸出 mu 和 logvar
mu     = Dense(latent_dim, name="enc_mu")(h)
logvar = Dense(latent_dim, name="enc_logvar")(h)

encoder = Model(inputs=[x_input, c_input], outputs=[mu, logvar], name="Encoder")

# -------------------------------
# 2. 定義 Decoder
# -------------------------------
# (A) 定義 z, c 輸入
z_input   = Input(shape=(latent_dim,),   name="z_input")
c_input_d = Input(shape=(num_classes,),  name="c_input_d")

# (B) 把 z 和 c 串接
zd = Concatenate(name="dec_concat")([z_input, c_input_d])  # → (None, latent_dim + num_classes)

# (C) 全連接層 fc_dec 與 LeakyReLU
d = Dense(256, name="dec_fc1")(zd)
d = LeakyReLU(alpha=0.2, name="dec_lrelu1")(d)
d = Dense(512, name="dec_fc2")(d)
d = LeakyReLU(alpha=0.2, name="dec_lrelu2")(d)
d = Dense(flattened_size, name="dec_fc3")(d)
d = LeakyReLU(alpha=0.2, name="dec_lrelu3")(d)

# (D) Reshape 成 (ds_h, ds_w, 256)
d = Reshape((ds_h, ds_w, 256), name="dec_reshape")(d)
# 現在 d.shape = (None, 4, 4, 256)

# (E) 三層 Conv2DTranspose + LeakyReLU （模擬 PyTorch ConvTranspose2d）
d = Conv2DTranspose(
    filters=64,
    kernel_size=3,
    strides=2,
    padding="same",
    name="dec_deconv1"
)(d)
d = LeakyReLU(alpha=0.2, name="dec_lrelu4")(d)
# d.shape = (None, 8, 8, 64)

d = Conv2DTranspose(
    filters=32,
    kernel_size=3,
    strides=2,
    padding="same",
    name="dec_deconv2"
)(d)
d = LeakyReLU(alpha=0.2, name="dec_lrelu5")(d)
# d.shape = (None, 16, 16, 32)

d = Conv2DTranspose(
    filters=in_ch,
    kernel_size=3,
    strides=2,
    padding="same",
    name="dec_deconv3"
)(d)
# d.shape = (None, 32, 32, in_ch=18)

# (F) 裁剪回 (29, 29) 大小
# d = Cropping2D(cropping=((0, 3), (0, 3)), name="dec_crop")(d)
# d.shape = (None, 29, 29, in_ch)

decoder = Model(inputs=[z_input, c_input_d], outputs=d, name="Decoder")

# -------------------------------
# 3. 建立 Full Model（mu → z → Decoder）
#    如果只想畫 Encoder 和 Decoder，各自分開 plot，可跳過這段
# -------------------------------
full_output = decoder([mu, c_input])  # mu 當作 z
full_model  = Model(inputs=[x_input, c_input], outputs=full_output, name="cVAE2d_Full")

# -------------------------------
# 4. 用 visualkeras 分別畫出 Encoder、Decoder、及 Full Model（如需要）。
# -------------------------------
if __name__ == "__main__":
    # 嘗試載入 TTF 字型，否則用 None
    try:
        font = ImageFont.truetype("../times-new-roman/times.ttf", size=60)
    except:
        font = None

    # --- 4.1 確保 Encoder build 完成，再畫 Encoder ---
    encoder.build(input_shape=[(None, H, W, in_ch), (None, num_classes)])
    img_enc = visualkeras.layered_view(
        encoder,
        legend=True,
        draw_volume=True,
        font=font,
        spacing=35,
        scale_xy=30,
        scale_z=0.5
    ).show()
    # img_enc.save("encoder_architecture.png")
    # print("Encoder 架構已儲存為 encoder_architecture.png")

    # --- 4.2 確保 Decoder build 完成，再畫 Decoder ---
    # 要先給予 decoder 一個假的輸入，才能產生完整 graph
    dummy_z = tf.zeros((1, latent_dim))
    dummy_c = tf.zeros((1, num_classes))
    _ = decoder([dummy_z, dummy_c])  # 觸發 build
    img_dec = visualkeras.layered_view(
        decoder,
        legend=True,
        draw_volume=True,
        font=font,
        spacing=35,
        scale_xy=30,
        scale_z=0.5
    ).show()
    # img_dec.save("decoder_architecture.png")
    # print("Decoder 架構已儲存為 decoder_architecture.png")

    # --- 4.3 如果想同時畫 Full Model（整條路徑），可以再解開下方註解 ---

    # full_model.build(input_shape=[(None, H, W, in_ch), (None, num_classes)])
    # img_full = visualkeras.layered_view(
    #     full_model,
    #     legend=True,
    #     draw_volume=True,
    #     font=font,
    #     spacing=35,
    #     scale_xy=30,
    #     scale_z=0.5
    # ).show()
    # img_full.save("cVAE2d_full_architecture.png")
    # print("Full Model 架構已儲存為 cVAE2d_full_architecture.png")




In [107]:
import math
import visualkeras
from PIL import ImageFont
import tensorflow as tf
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    Conv2DTranspose,
    LeakyReLU,
    Flatten,
    Dense,
    Concatenate,
    Reshape
)
from tensorflow.keras.models import Model

# -------------------------------
# 參數設定（請依實際需求調整）
# -------------------------------
in_ch       = 18    # 輸入通道數
latent_dim  = 50    # 潛在空間維度
num_classes = 2     # 條件標籤數量
H, W        = 29, 29

# 經過三次 stride=2 後的 feature map 長寬
ds_h = math.ceil(H / 8)  # ceil(29/8) = 4
ds_w = math.ceil(W / 8)  # = 4
flattened_size = 256 * ds_h * ds_w  # 256 * 4 * 4 = 4096

# -------------------------------
# 1. 定義統一的 color_map
#    注意：每個 Layer Class 對應一個字典，包含 'fill' (填充色) 以及可選的 'outline'
# -------------------------------
color_map = {
    tf.keras.layers.Conv2D:          {"fill": "lightblue"},
    tf.keras.layers.Conv2DTranspose: {"fill": "lightblue"},
    tf.keras.layers.LeakyReLU:       {"fill": "lightgreen"},
    tf.keras.layers.Dense:           {"fill": "orange"},
    tf.keras.layers.Flatten:         {"fill": "gray"},
    tf.keras.layers.Concatenate:     {"fill": "mediumpurple"},
    tf.keras.layers.Reshape:         {"fill": "#555555"},
    # 若有其他層要自訂顏色，可再加：
    # tf.keras.layers.BatchNormalization: {"fill": "pink"},
}

# -------------------------------
# 2. 定義 Encoder
# -------------------------------
x_input = Input(shape=(H, W, in_ch), name="x_input")
c_input = Input(shape=(num_classes,), name="c_input")

h = Conv2D(64, kernel_size=3, strides=2, padding="same", name="enc_conv1")(x_input)
h = LeakyReLU(alpha=0.2, name="enc_lrelu1")(h)

h = Conv2D(128, kernel_size=3, strides=2, padding="same", name="enc_conv2")(h)
h = LeakyReLU(alpha=0.2, name="enc_lrelu2")(h)

h = Conv2D(256, kernel_size=3, strides=2, padding="same", name="enc_conv3")(h)
h = LeakyReLU(alpha=0.2, name="enc_lrelu3")(h)
# h.shape = (None, 4, 4, 256)

h = Flatten(name="enc_flatten")(h)  # → (None, 4096)
h = Concatenate(name="enc_concat")([h, c_input])  # → (None, 4096 + num_classes)

h = Dense(512, name="enc_fc1")(h)
h = LeakyReLU(alpha=0.2, name="enc_lrelu4")(h)
h = Dense(256, name="enc_fc2")(h)
h = LeakyReLU(alpha=0.2, name="enc_lrelu5")(h)

mu     = Dense(latent_dim, name="enc_mu")(h)
logvar = Dense(latent_dim, name="enc_logvar")(h)

encoder = Model(inputs=[x_input, c_input], outputs=[mu, logvar], name="Encoder")

# -------------------------------
# 3. 定義 Decoder（已移除 Cropping2D）
# -------------------------------
z_input   = Input(shape=(latent_dim,),  name="z_input")
c_input_d = Input(shape=(num_classes,), name="c_input_d")

zd = Concatenate(name="dec_concat")([z_input, c_input_d])  # → (None, latent_dim + num_classes)

d = Dense(256, name="dec_fc1")(zd)
d = LeakyReLU(alpha=0.2, name="dec_lrelu1")(d)
d = Dense(512, name="dec_fc2")(d)
d = LeakyReLU(alpha=0.2, name="dec_lrelu2")(d)
d = Dense(flattened_size, name="dec_fc3")(d)
d = LeakyReLU(alpha=0.2, name="dec_lrelu3")(d)

d = Reshape((ds_h, ds_w, 256), name="dec_reshape")(d)
# d.shape = (None, 4, 4, 256)

d = Conv2DTranspose(64, kernel_size=3, strides=2, padding="same", name="dec_deconv1")(d)
d = LeakyReLU(alpha=0.2, name="dec_lrelu4")(d)
# d.shape = (None, 8, 8, 64)

d = Conv2DTranspose(32, kernel_size=3, strides=2, padding="same", name="dec_deconv2")(d)
d = LeakyReLU(alpha=0.2, name="dec_lrelu5")(d)
# d.shape = (None, 16, 16, 32)

d = Conv2DTranspose(in_ch, kernel_size=3, strides=2, padding="same", name="dec_deconv3")(d)
# d.shape = (None, 32, 32, in_ch)

# 直接輸出 (32, 32, in_ch)
decoder = Model(inputs=[z_input, c_input_d], outputs=d, name="Decoder")

# -------------------------------
# 4. （可選）建立 Full Model
# -------------------------------
full_output = decoder([mu, c_input])
full_model  = Model(inputs=[x_input, c_input], outputs=full_output, name="cVAE2d_Full")

# -------------------------------
# 5. 用 visualkeras 分別畫出 Encoder、Decoder，並共用 color_map
# -------------------------------
if __name__ == "__main__":
    # 嘗試載入 TTF 字型，若失敗設為 None
    try:
        font = ImageFont.truetype("../times-new-roman/times.ttf", size=60)
    except:
        font = None

    # ---- 5.1 畫 Encoder ----
    encoder.build(input_shape=[(None, H, W, in_ch), (None, num_classes)])
    img_enc = visualkeras.layered_view(
        encoder,
        legend=True,          # 顯示圖例
        draw_volume=True,     # 嘗試畫出卷積層的厚度
        font=font,
        spacing=35,
        scale_xy=30,
        scale_z=0.5,
        color_map=color_map   # 使用統一配色
    ).show()
    # img_enc.save("encoder_architecture.png")
    # print("Encoder 架構已儲存為 encoder_architecture.png")


    # ---- 5.2 畫 Decoder ----
    dummy_z = tf.zeros((1, latent_dim))
    dummy_c = tf.zeros((1, num_classes))
    _ = decoder([dummy_z, dummy_c])  # 觸發 build
    img_dec = visualkeras.layered_view(
        decoder,
        legend=True,
        draw_volume=True,
        font=font,
        spacing=35,
        scale_xy=30,
        scale_z=0.5,
        color_map=color_map   # 同樣使用相同配色
    ).show()
    # img_dec.save("decoder_architecture.png")
    # print("Decoder 架構已儲存為 decoder_architecture.png")


    # ---- 5.3 （可選）畫 Full Model，也使用相同配色 ----
    """
    full_model.build(input_shape=[(None, H, W, in_ch), (None, num_classes)])
    img_full = visualkeras.layered_view(
        full_model,
        legend=True,
        draw_volume=True,
        font=font,
        spacing=35,
        scale_xy=30,
        scale_z=0.5,
        color_map=color_map
    )
    img_full.save("cVAE2d_full_architecture.png")
    print("Full Model 架構已儲存為 cVAE2d_full_architecture.png")
    img_full.show()
    """
